# Spaceship Titanic: Final CatBoost Pipeline

This notebook contains the final selected CatBoost-based solution for the Kaggle Spaceship Titanic classification task. It focuses on a stable feature-engineering pipeline, two CatBoost model variants, 5-fold stratified cross-validation, out-of-fold blend selection, and final submission generation.

Expected input files in the same directory:

- `train.csv`
- `test.csv`
- `sample_submission.csv`

Main output file:

- `catboost_last_try_submission.csv`


## 1. Environment and Imports

In [ ]:
# Install CatBoost first if it is not available in your environment.
# !pip install catboost -q

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from catboost import CatBoostClassifier

RANDOM_STATE = 42
N_SPLITS = 5

TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"
SAMPLE_SUBMISSION_PATH = "sample_submission.csv"
OUTPUT_PATH = "catboost_last_try_submission.csv"
OOF_PROB_PATH = "catboost_last_try_oof_probabilities.csv"
TEST_PROB_PATH = "catboost_last_try_test_probabilities.csv"


## 2. Load Data

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("Training data shape:", train_df.shape)
print("Test data shape:", test_df.shape)
train_df.head()

## 3. Feature Engineering

The feature-engineering step extracts group information from `PassengerId`, cabin structure from `Cabin`, surname information from `Name`, spending aggregates, age-related indicators, and interaction features for the second CatBoost model.


In [ ]:
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """Create structured and interaction features for the Spaceship Titanic dataset."""
    df = df.copy()

    # Passenger group information
    df[["GroupID", "GroupMember"]] = df["PassengerId"].str.split("_", expand=True)
    df["GroupID"] = pd.to_numeric(df["GroupID"], errors="coerce")
    df["GroupMember"] = pd.to_numeric(df["GroupMember"], errors="coerce")

    # Cabin structure
    df[["Deck", "CabinNum", "Side"]] = df["Cabin"].fillna("Unknown/0/U").str.split("/", expand=True)
    df["CabinNum"] = pd.to_numeric(df["CabinNum"], errors="coerce")

    # Surname signal
    df["Surname"] = (
        df["Name"]
        .fillna("Unknown Unknown")
        .apply(lambda x: str(x).split()[-1] if len(str(x).split()) > 0 else "Unknown")
    )

    # Spending features
    spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
    for col in spend_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["TotalSpend"] = df[spend_cols].sum(axis=1)
    df["NoSpend"] = (df["TotalSpend"] == 0).astype(int)
    df["LuxurySpend"] = df["Spa"].fillna(0) + df["VRDeck"].fillna(0)
    df["BasicSpend"] = (
        df["RoomService"].fillna(0)
        + df["FoodCourt"].fillna(0)
        + df["ShoppingMall"].fillna(0)
    )

    df["Spa_ratio"] = df["Spa"] / (df["TotalSpend"] + 1)
    df["VRDeck_ratio"] = df["VRDeck"] / (df["TotalSpend"] + 1)
    df["FoodCourt_ratio"] = df["FoodCourt"] / (df["TotalSpend"] + 1)
    df["Luxury_ratio"] = df["LuxurySpend"] / (df["TotalSpend"] + 1)

    # Age features
    df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
    df["AgeBand"] = pd.cut(
        df["Age"],
        bins=[-1, 12, 18, 25, 40, 60, 100],
        labels=["Child", "Teen", "Youth", "Adult", "MidAge", "Senior"],
    )

    df["IsChild"] = (df["Age"] < 18).astype(float)
    df["IsAdult"] = (df["Age"] >= 18).astype(float)

    # Boolean columns are kept as categorical-like objects for CatBoost.
    for col in ["CryoSleep", "VIP"]:
        df[col] = df[col].astype("object")

    # Group-level features
    df["GroupSize"] = df.groupby("GroupID")["PassengerId"].transform("count")
    df["IsAlone"] = (df["GroupSize"] == 1).astype(int)

    # Interaction features
    df["Deck_Side"] = df["Deck"].astype(str) + "_" + df["Side"].astype(str)
    df["HomePlanet_Deck"] = df["HomePlanet"].astype(str) + "_" + df["Deck"].astype(str)
    df["Destination_Deck"] = df["Destination"].astype(str) + "_" + df["Deck"].astype(str)
    df["VIP_HomePlanet"] = df["VIP"].astype(str) + "_" + df["HomePlanet"].astype(str)
    df["Cryo_NoSpend"] = df["CryoSleep"].astype(str) + "_" + df["NoSpend"].astype(str)

    return df


In [ ]:
train_fe = feature_engineering(train_df)
test_fe = feature_engineering(test_df)

target = train_fe["Transported"].astype(int)
train_fe = train_fe.drop(columns=["Transported"])
test_passenger_ids = test_fe["PassengerId"]

print("Feature-engineered training data shape:", train_fe.shape)
print("Feature-engineered test data shape:", test_fe.shape)

## 4. Define Feature Sets

Two feature sets are used:

- **Model A**: a compact set of core numerical and categorical features.
- **Model B**: an extended set with interaction and ratio features.

The final prediction is a weighted blend of these two CatBoost models.


In [ ]:
base_features = [
    "HomePlanet", "CryoSleep", "Destination", "VIP",
    "Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck",
    "GroupID", "GroupMember", "Deck", "CabinNum", "Side",
    "Surname", "TotalSpend", "NoSpend", "GroupSize", "IsAlone",
    "AgeBand", "LuxurySpend", "BasicSpend",
]

cross_features = [
    "HomePlanet", "CryoSleep", "Destination", "VIP",
    "Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck",
    "GroupID", "GroupMember", "Deck", "CabinNum", "Side",
    "Surname", "TotalSpend", "NoSpend", "GroupSize", "IsAlone",
    "AgeBand", "LuxurySpend", "BasicSpend",
    "Deck_Side", "HomePlanet_Deck", "Destination_Deck", "VIP_HomePlanet",
    "Cryo_NoSpend", "Spa_ratio", "VRDeck_ratio", "FoodCourt_ratio", "Luxury_ratio",
    "IsChild", "IsAdult",
]

X_A = train_fe[base_features].copy()
X_test_A = test_fe[base_features].copy()

X_B = train_fe[cross_features].copy()
X_test_B = test_fe[cross_features].copy()

print("Model A feature count:", len(base_features))
print("Model B feature count:", len(cross_features))

## 5. Prepare Features for CatBoost

Categorical features are converted to strings and missing numerical values are filled using the training median. This keeps preprocessing simple and reproducible while still allowing CatBoost to handle categorical variables directly.


In [ ]:
def prepare_for_catboost(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
):
    """Prepare categorical and numerical columns for CatBoost."""
    X_train = X_train.copy()
    X_test = X_test.copy()

    categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
    numerical_cols = [col for col in X_train.columns if col not in categorical_cols]

    for col in categorical_cols:
        X_train[col] = X_train[col].astype("object").fillna("Missing").astype(str)
        X_test[col] = X_test[col].astype("object").fillna("Missing").astype(str)

    for col in numerical_cols:
        median_value = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_value)
        X_test[col] = X_test[col].fillna(median_value)

    cat_feature_indices = [X_train.columns.get_loc(col) for col in categorical_cols]

    return X_train, X_test, categorical_cols, numerical_cols, cat_feature_indices


X_A, X_test_A, cat_cols_A, num_cols_A, cat_idx_A = prepare_for_catboost(X_A, X_test_A)
X_B, X_test_B, cat_cols_B, num_cols_B, cat_idx_B = prepare_for_catboost(X_B, X_test_B)

print("Model A categorical columns:", cat_cols_A)
print("Model B categorical columns:", cat_cols_B)

## 6. Cross-validated CatBoost Training

In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

oof_A = np.zeros(len(X_A))
oof_B = np.zeros(len(X_B))

test_pred_A = np.zeros(len(X_test_A))
test_pred_B = np.zeros(len(X_test_B))

fold_records = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_A, target), 1):
    print(f"========== Fold {fold} ==========")

    X_train_A, X_valid_A = X_A.iloc[train_idx], X_A.iloc[valid_idx]
    X_train_B, X_valid_B = X_B.iloc[train_idx], X_B.iloc[valid_idx]
    y_train, y_valid = target.iloc[train_idx], target.iloc[valid_idx]

    model_A = CatBoostClassifier(
        iterations=2500,
        learning_rate=0.03,
        depth=6,
        loss_function="Logloss",
        eval_metric="Accuracy",
        random_seed=42 + fold,
        verbose=0,
    )

    model_B = CatBoostClassifier(
        iterations=3000,
        learning_rate=0.025,
        depth=7,
        loss_function="Logloss",
        eval_metric="Accuracy",
        random_seed=100 + fold,
        verbose=0,
    )

    model_A.fit(
        X_train_A,
        y_train,
        cat_features=cat_idx_A,
        eval_set=(X_valid_A, y_valid),
        use_best_model=True,
    )

    model_B.fit(
        X_train_B,
        y_train,
        cat_features=cat_idx_B,
        eval_set=(X_valid_B, y_valid),
        use_best_model=True,
    )

    valid_pred_A = model_A.predict_proba(X_valid_A)[:, 1]
    valid_pred_B = model_B.predict_proba(X_valid_B)[:, 1]

    oof_A[valid_idx] = valid_pred_A
    oof_B[valid_idx] = valid_pred_B

    test_pred_A += model_A.predict_proba(X_test_A)[:, 1] / N_SPLITS
    test_pred_B += model_B.predict_proba(X_test_B)[:, 1] / N_SPLITS

    acc_A = accuracy_score(y_valid, (valid_pred_A >= 0.5).astype(int))
    acc_B = accuracy_score(y_valid, (valid_pred_B >= 0.5).astype(int))

    fold_records.append({"fold": fold, "model_A_accuracy": acc_A, "model_B_accuracy": acc_B})

    print(f"Model A fold accuracy: {acc_A:.5f}")
    print(f"Model B fold accuracy: {acc_B:.5f}")
    print()

fold_results = pd.DataFrame(fold_records)
fold_results

## 7. Blend Weight Selection

The blend weight is selected using out-of-fold predictions. The threshold is fixed at `0.5` to match the final selected submission strategy.


In [ ]:
best_score = -1
best_weight = None

for weight_A in np.arange(0.0, 1.01, 0.02):
    blend_oof = weight_A * oof_A + (1 - weight_A) * oof_B
    blend_pred = (blend_oof >= 0.5).astype(int)
    score = accuracy_score(target, blend_pred)

    if score > best_score:
        best_score = score
        best_weight = weight_A

print("Best OOF accuracy:", round(best_score, 5))
print("Best weight for Model A:", round(best_weight, 2))
print("Best weight for Model B:", round(1 - best_weight, 2))

## 8. Generate Submission File

In [ ]:
final_oof_prob = best_weight * oof_A + (1 - best_weight) * oof_B
final_test_prob = best_weight * test_pred_A + (1 - best_weight) * test_pred_B
final_test_pred = final_test_prob >= 0.5

submission = pd.DataFrame({
    "PassengerId": test_passenger_ids,
    "Transported": final_test_pred,
})

submission.to_csv(OUTPUT_PATH, index=False)

# Probability files are saved for reproducibility and later analysis.
pd.DataFrame({
    "PassengerId": train_df["PassengerId"],
    "Transported": target,
    "Model_A_Probability": oof_A,
    "Model_B_Probability": oof_B,
    "Blended_Probability": final_oof_prob,
}).to_csv(OOF_PROB_PATH, index=False)

pd.DataFrame({
    "PassengerId": test_passenger_ids,
    "Model_A_Probability": test_pred_A,
    "Model_B_Probability": test_pred_B,
    "Blended_Probability": final_test_prob,
}).to_csv(TEST_PROB_PATH, index=False)

print(f"Saved submission file: {OUTPUT_PATH}")
print(f"Saved OOF probability file: {OOF_PROB_PATH}")
print(f"Saved test probability file: {TEST_PROB_PATH}")
submission.head()

## 9. Reproducibility Notes

- The final Kaggle submission uses the `catboost_last_try_submission.csv` file generated above.
- The model uses 5-fold stratified cross-validation.
- CatBoost handles categorical features directly after missing values are converted to the string value `Missing`.
- The final prediction threshold is fixed at `0.5`.
- Additional probability files are exported to make later analysis and comparison easier.
